Faunimo : l'ouvrier Colab (session Puissance, 27/10). Une 3e carte pour les photos Godot, à côté des deux de Kaggle.

Arthur, une seule fois : dans Colab, la clé « Secrets » (à gauche), ajouter KAGGLE_USERNAME (masrox) et KAGGLE_KEY
(la « key » de kaggle.json, créé sur kaggle.com/settings > Create New Token), « Accès au carnet » coché.
Puis : Exécution > Modifier le type d'exécution > GPU T4, et Exécution > Tout exécuter. L'onglet reste ouvert.

Le carnet tire le jeu (masrox/faunimo-jeu) une fois, puis attend les lots que le PC dépose dans masrox/faunimo-colab-lot
(atelier_colab.py) ; chaque lot passe par le même carnet.py que Kaggle, et sa sortie repart dans masrox/faunimo-colab-sortie.
Aucune clé n'est écrite ni affichée : elle ne vit que dans les variables d'environnement de cette séance.

In [ ]:
import glob
import json
import os
import shutil
import subprocess
import time
import zipfile

from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
subprocess.run(["pip", "install", "-q", "kaggle"], check=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"], capture_output=True, text=True).stdout)

LOT, SORTIE, JEU = "masrox/faunimo-colab-lot", "masrox/faunimo-colab-sortie", "masrox/faunimo-jeu"


def k(*args, delai=3600):
    r = subprocess.run(["kaggle"] + list(args), capture_output=True, text=True, timeout=delai)
    return r.returncode, (r.stdout or "") + (r.stderr or "")


def deplier(dossier):
    """Kaggle rend les dossiers en .zip (dir-mode zip) : on les déplie sur place, jusqu'au bout."""
    while True:
        zips = glob.glob(dossier + "/**/*.zip", recursive=True)
        if not zips:
            return
        for z in zips:
            cible = os.path.splitext(z)[0]                 # faunimo.zip -> faunimo/, faunimo-jeu.zip -> faunimo-jeu/
            with zipfile.ZipFile(z) as f:
                f.extractall(cible)
            os.remove(z)
            seul = os.listdir(cible)                       # le zip contenait déjà son dossier : faunimo/faunimo -> faunimo
            if len(seul) == 1 and seul[0] == os.path.basename(cible) and os.path.isdir(os.path.join(cible, seul[0])):
                tmp = cible + "-tmp"
                os.rename(os.path.join(cible, seul[0]), tmp)
                os.rmdir(cible)
                os.rename(tmp, cible)


# 1. le jeu, une fois par séance (environ 8 Go)
if not glob.glob("/donnees/faunimo-jeu/art/modeles"):
    os.makedirs("/content/jeu", exist_ok=True)
    print("jeu :", k("datasets", "download", "-d", JEU, "-p", "/content/jeu", "--unzip")[1][-300:])
    deplier("/content/jeu")
    trouve = [os.path.dirname(os.path.dirname(p)) for p in glob.glob("/content/jeu/**/art/modeles", recursive=True)]
    os.makedirs("/donnees", exist_ok=True)
    if trouve and not os.path.exists("/donnees/faunimo-jeu"):
        os.symlink(trouve[0], "/donnees/faunimo-jeu")
print("gros fichiers :", os.path.realpath("/donnees/faunimo-jeu"))

# 2. la boucle : un lot à la fois, tant que la séance dure
faits = set()
creee = k("datasets", "status", SORTIE)[0] == 0 and "ready" in k("datasets", "status", SORTIE)[1]
while True:
    code, liste = k("datasets", "files", LOT, "--csv", delai=300)
    lots = [l.split(",")[0][4:-4] for l in liste.splitlines() if l.startswith("lot-") and l.split(",")[0].endswith(".txt")]
    neufs = [l for l in lots if l not in faits]
    if not neufs:
        time.sleep(60)
        continue
    lot = neufs[0]
    print(time.strftime("%H:%M"), "lot", lot, flush=True)
    shutil.rmtree("/content/lot", ignore_errors=True)
    os.makedirs("/content/lot")
    print(k("datasets", "download", "-d", LOT, "-p", "/content/lot", "--unzip")[1][-200:])
    deplier("/content/lot")
    carnets = glob.glob("/content/lot/**/carnet.py", recursive=True)
    sortie = "/content/sortie-" + lot
    shutil.rmtree(sortie, ignore_errors=True)
    env = dict(os.environ, BIBELOTS_JEU_GROS="/donnees/faunimo-jeu")
    racine = os.path.dirname(glob.glob("/content/lot/**/travaux.json", recursive=True)[0])
    r = subprocess.run(["python", carnets[0], "--entree", racine, "--sortie", sortie, "--base", "/content/base"], env=env)
    print("carnet : code", r.returncode, flush=True)
    envoi = "/content/envoi"
    shutil.rmtree(envoi, ignore_errors=True)
    os.makedirs(envoi)
    shutil.make_archive(os.path.join(envoi, lot), "zip", sortie)
    open(os.path.join(envoi, "fait-%s.txt" % lot), "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))
    json.dump({"title": "Faunimo colab sortie", "id": SORTIE, "licenses": [{"name": "other"}]},
              open(os.path.join(envoi, "dataset-metadata.json"), "w"))
    if creee:
        print(k("datasets", "version", "-p", envoi, "-m", lot, "-q")[1][-200:])
    else:
        print(k("datasets", "create", "-p", envoi, "-q")[1][-200:])
        creee = True
    faits.add(lot)
